# Classic Control
---

Kyle Hellstrom (22343261), David Bracken (22342044)

---
## 1. Why Reinforcement Learning is the machine learning paradigm of choice for this task
Reinforcement learning is ideal for the MountainCar gymnasium environment as it allows an agent to learn through actions made to the environment through trial and error to maximise the reward returned.

The other machine learning paradigms would not work as well since supervised learning requires labeled data but there is nothing labelling ideal actions per state which is why RL's exploration is chosen. Unsupervised learning couldnt be used since the mountain car does not have unlabeled data but actions and states which is how RL interacts with an environment.

## 2. The Gym Environment
For this implementation, out of the 4 available classic control environments on "https://gymnasium.farama.org/environments/classic_control/" excluding cart-pole as per the requirements, we used the MountainCar-v0 environment where the given parameters and settings are as follows:
- Action Space: Discrete(3)
    - There are 3 discrete deterministic actions:
        - 0: Accelerate to the left
        - 1: Don’t accelerate
        - 2: Accelerate to the right

- Observation Space: Box([-1.2 -0.07], [0.6 0.07], (2,), float32)
    - The observation is a ndarray with shape (2,) where the elements correspond to the following:
        | Num | Observation                            | Min   | Max   | Unit        |
        |-----|----------------------------------------|-------|-------|-------------|
        | 0   | position of the car along the x-axis   | -1.2  | 0.6   | position (m)|
        | 1   | velocity of the car                    | -0.07 | 0.07  | velocity (v)|

- Episode Length: 
    - 200 Steps and will terminate if not done

- Default Reward:
    - -1 per step, agent focuses on speed

- Preprocessing:
    - Seed is set for cpu and/or GPU
    - Each step has the observation state normalised between [0,1]

- Processing:
    - Q-Network takes in observations to predict q values for each action
    - Choises are made using e-greedy

The challenge of this environment is mainly because of the rewards where the agent has to learn momentum and go backwards in order to get higher along the x axis

## 3. Implementation
### Imports

In [ ]:
import random, collections
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
import gymnasium as gym
import numpy as np

# Numpy bool error solution
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

### Hyperparameters

In [ ]:
TRAIN = False
RENDER = not TRAIN

NUM_AGENTS = 4
NUM_EPISODES = 500
MAX_STEPS = 200
BATCH_SIZE = 64
DISCOUNT = 0.99
LEARNING_RATE = 1e-4
# LR_DECAY = 0.001
BUFFER_SIZE = 50000
MIN_REPLAY_SIZE = 5000
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 0.995
TAU = 0.001


### Seed

In [ ]:
SEED = 21
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

### Environment

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = gym.make("MountainCar-v0", render_mode="human" if RENDER else None)

### Normalise Function

In [ ]:
def Observe_state(state):
    low = env.observation_space.low
    high = env.observation_space.high
    # print(f"State: {state} | Low: {low} | High: {high}")
    output = (state - low) / (high - low)
    # print(f"Out: {output}")
    return output

### Custom Reward

In [ ]:
def custom_reward(next_state, env_reward=0.0):
    position, velocity = next_state
    # Env reward is -1 for each step
    reward = env_reward
    # Velocity
    reward += 5 * abs(velocity)
    if position >= 0.5:
        reward += 50

    # print(f"Env Reward: {env_reward} | Mod Reward: {reward}")
    return reward

### Custom Step and reset

In [ ]:
def custom_step(action):
    raw_next_state, env_reward, terminated, truncated, info = env.step(action)
    mod_reward = custom_reward(raw_next_state, env_reward)
    norm_next_state = Observe_state(raw_next_state)
    return norm_next_state, mod_reward, terminated, truncated, info

def custom_reset(seed=None):
    raw_state, info = env.reset(seed=seed)
    return Observe_state(raw_state), info

### Replay buffer

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        state, action , reward, next_state, done = zip(*batch)
        return (
            torch.tensor(np.stack(state), dtype=torch.float32, device=DEVICE),
            torch.tensor(action, dtype=torch.long, device=DEVICE),
            torch.tensor(reward, dtype=torch.float32, device=DEVICE),
            torch.tensor(np.stack(next_state), dtype=torch.float32, device=DEVICE),
            torch.tensor(done, dtype=torch.float32, device=DEVICE),
        )

    def __len__(self):
        return len(self.buffer)

### Q Network

In [ ]:
class QNet(nn.Module):    
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 25)
        self.fc2 = nn.Linear(25, 20)
        self.fc3 = nn.Linear(20, action_dim)

    def forward(self, x):
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        return self.fc3(x)

### Agent

In [ ]:
class DQNAgent:
    def __init__(self, env, replay_buffer):
        obs_dim = env.observation_space.shape[0]
        n_actions = env.action_space.n

        self.online = QNet(obs_dim, n_actions).to(DEVICE)
        self.target = QNet(obs_dim, n_actions).to(DEVICE)
        self.target.load_state_dict(self.online.state_dict())
        self.optimizer = optim.Adam(
            self.online.parameters(),
            lr=LEARNING_RATE,
            # weight_decay=LR_DECAY
        )

        self.replay = replay_buffer
        self.steps = 0

    def soft_update(self):
        for target_net, online_net in zip(self.target.parameters(), self.online.parameters()):
            target_net.data.copy_(online_net.data * TAU + target_net.data * (1 - TAU))

    def learn(self, batch_size):
        states, actions, rewards, next_states, dones = self.replay.sample(batch_size)
        actions_idx = actions.unsqueeze(1)          
        rewards = rewards.unsqueeze(1)              
        dones = dones.unsqueeze(1)                  

        q_values = self.online(states).gather(1, actions_idx) 

        # Double DQN target
        with torch.no_grad():
            next_actions = self.online(next_states).argmax(dim=1, keepdim=True) 
            next_q = self.target(next_states).gather(1, next_actions)           
            target_q = rewards + (1.0 - dones) * DISCOUNT * next_q              

        loss = nn.SmoothL1Loss()(q_values, target_q)

        # Backprop
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.online.parameters(), 5)
        self.optimizer.step()

        self.steps += 1
        self.soft_update()
        
        # # Hard update - Didnt work as good as soft update when testing
        # if self.steps % 1000 == 0:
        #     self.target.load_state_dict(self.online.state_dict())
        
        return loss.item()

### Multi-Agent Act

In [ ]:
def multi_agent_act(agents, state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()

    with torch.no_grad():
        state_v = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        qs = [agent.online(state_v) for agent in agents]
        mean_q = torch.mean(torch.stack(qs), dim=0)
        return int(mean_q.argmax(dim=1).item())

Create agents and shared replay buffer

In [ ]:
shared_replay = ReplayBuffer(BUFFER_SIZE)
agents = [DQNAgent(env, shared_replay) for _ in range(NUM_AGENTS)]

### Pre-fill replay buffer

In [ ]:
def buffer_prefill():
    state, _ = custom_reset(seed=SEED)
    for _ in range(MIN_REPLAY_SIZE):
        action = env.action_space.sample()
        next_state, reward, terminated, truncated, _ = custom_step(action)
        shared_replay.push(state, action, reward, next_state, terminated)
        if terminated:
            state, _ = custom_reset()
        else:
            state = next_state

### Plots

In [ ]:
def plot_training(rewards, losses, average=50, max_reward=1000):
    rewards = np.array(rewards)
    rewards_history = np.clip(rewards, None, max_reward)

    if len(rewards) >= average:
        sma = np.clip(
            np.convolve(rewards, np.ones(average) / average, mode="valid"),
            None,
            max_reward
        )
    else:
        sma = None

    plt.figure()
    plt.title(f"Rewards")
    plt.plot(rewards_history, label="Raw Reward")
    if sma is not None:
        plt.plot(sma, label=f"SMA {average}")
    plt.xlabel("Episode")
    plt.ylabel("Rewards")
    plt.legend()
    plt.savefig("Results/reward_plot.png", dpi=600, bbox_inches="tight")
    plt.show()

    plt.figure()
    plt.title(f"Loss")
    plt.plot(losses, label="Loss")
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.savefig("Results/loss_plot.png", dpi=600, bbox_inches="tight")
    plt.show()

### Training loop

In [ ]:
def train():
    epsilon = EPS_START
    rewards_history = []
    loss_history = []

    for episode in range(1, NUM_EPISODES + 1):
        state, _ = custom_reset()
        ep_reward = 0.0
        ep_steps = 0

        for step in range(MAX_STEPS):
            action = multi_agent_act(agents, state, epsilon)
            next_state, reward, terminated, truncated, _ = custom_step(action)

            shared_replay.push(state, action, reward, next_state, terminated)
            state = next_state
            ep_reward += reward
            ep_steps += 1

            # 4 is learning delay
            if step % 4 == 0 and len(shared_replay) >= BATCH_SIZE:
                for agent in agents:
                    loss = agent.learn(BATCH_SIZE)
                    loss_history.append(loss)

            if terminated:
                break

        rewards_history.append(ep_reward)
        epsilon = max(EPS_END, epsilon * EPS_DECAY)

        if ep_steps < 200 or episode % 10 == 0:
        # if ep_steps < 200 or episode % 100 == 0:
            print(
                f"Episode {episode:4d} | "
                f"Reward {ep_reward:7.2f} | "
                f"Steps {ep_steps:3d} | "
                f"Epsilon {epsilon:.3f} | "
            )

    torch.save(agents[0].online.state_dict(), f"Results/dqn_mountaincar_{NUM_EPISODES}.pth")
    print("Model saved.")
    plot_training(rewards_history, loss_history)

### Test

In [ ]:
def test(episodes):
    for episode in range(episodes):
        state, _ = custom_reset(seed=SEED)
        episode_reward = 0.0
        ep_steps = 0

        for _ in range(MAX_STEPS):
            action = multi_agent_act(agents, state, epsilon=0.0)
            next_state, reward, terminated, truncated, _ = custom_step(action)

            state = next_state
            episode_reward += reward
            ep_steps += 1
            if terminated:
                break
        print(
            f"Episode {episode+1:3d} | "
            f"Steps {ep_steps:3d} | "
            f"Return {episode_reward:7.2f}"
        )
    env.close()

### Main Runner
Change the ```TRAIN``` constant in the hyperparameters to choose to train or test.

The test file will be the file in the same dir as the file and will be chosen based off the `NUM_EPISODES` constant.

In [ ]:
if __name__ == "__main__":
    if TRAIN:
        buffer_prefill()
        train()
    else:
        for agent in agents:
            agent.online.load_state_dict(torch.load(f"Results/dqn_mountaincar_{NUM_EPISODES}.pth", map_location=DEVICE))
            agent.target.load_state_dict(agent.online.state_dict())
        test(2)

## 4. Independently researched concepts
- Custom Rewards
    - We kept the original environment reward of -1 per step but included additional rewards
    - Increased reward based the velocity of the cart per step 
    - A bonus reward for reaching termination

- Soft Update
    - The hard update of weights every 1000 steps gave worse results then using the soft update method 
    - Soft Update Implementation
    ![Soft Update Implementation](Resources/soft_reward_plot.png)    
    
    - Hard Update Implementation
    ![Hard Update Implementation](Resources/hard_reward_plot.png)

- Double DQN
    - Implemented to prevent maximisation bias by decoupling action selection and evaluation

- Multi-Agent Training
    - Instead of just using the values from 1 agent, multiple agents are used and results averaged to attempt to remove outliers

- Additional unused features:
    - Learning rate decay from 'https://ai.stackexchange.com/questions/28079/deep-q-learning-catastrophic-drop-reasons' which was tested but failed to return successful outputs.

## 5. References
- https://gymnasium.farama.org/environments/classic_control/mountain_car/
- https://www.akshaymakes.com/blogs/deep_q_learning
- https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html
- https://github.com/udacity/deep-reinforcement-learning/blob/master/dqn/README.md
- CS4287-P-RL_DQN_Cartpole_Listing.pdf
- https://stackoverflow.com/questions/79058350/module-numpy-has-no-attribute-bool8-in-cartpole-problem-openai-gym
- https://greentec.github.io/reinforcement-learning-third-en/#soft-update-target-network